# Code-based cryptography: Exam 4
February 20th, Thursday

### Duration: 1 hour and 30 minutes.
Extra time (special arrangements): + 30 minutes $\rightarrow$ 2 hours in total.

<br>

Recall that the weight enumerator of a code $C$ is
$W_C(x)=\sum_{i=0}^n w_C(i) x^i$ where  $w_C(i)=|\{ c \in C : w(c)=i\}|$. We always have $w_C(0)=1$. In particular, if $C$ is the zero code, then $W_C(x)=1$.

*N.B.: SageMaths has a procedure to compute the weight distribution but **only for nonzero codes** on old versions. We will then use the following fix in this case.*


# Fait avec Baptiste Thierry

In [143]:
def weight_enumerator_fixed(C):
    if C.dimension() == 0 :
        return 1
    else:
        return C.weight_enumerator(bivariate=False)

## 1. Behaviour of the dimension of the hull of random codes
**Q1.** Write a function ``hull`` that takes a code $C$ as input and return its hull $\mathcal{H}(C) = C \cap C^\perp$ as a linear code, using the generator and the parity check matrices of the input code.

You can either
1. use ``V1.intersection(V2)`` that computes the intersection of the linear spaces (beware, it may not work directly on linear codes with prehistoric versions of SageMath) with ``C.dual_code()``,
2. or compute directly $\mathcal{H}(C)$ as the kernel of a matrix that depends on some generator matrix and parity check matrix of $C$.


*In both cases, you may use ``codes.LinearCode(V)``, which returns the linear code associated to the vector space V **if V is not zero**. You can return the zero code otherwise by using `codes.random_linear_code(F,n,0)`.*

In [144]:
def hull(C):
    dual = C.dual_code()
    H=C.parity_check_matrix()
    Hdual = dual.parity_check_matrix()
    F=C.base_field()
    n=C.length()
    V=kernel(H.transpose())
    Z=kernel(Hdual.transpose())
    respespace = (V.intersection(Z))
    if respespace.dimension()==0:
        resp=codes.random_linear_code(F,n,0)
    else :
        resp= codes.LinearCode(V.intersection(Z))
    return resp
    
    

For sanity check, run the following cell. If it does not pass, you should worry.

In [145]:
C = codes.random_linear_code(GF(4), 11, 7)
hC=hull(C)
print(hC)
assert hC.is_subcode(C) and hC.is_subcode(C.dual_code())

[11, 0] linear code over GF(4)


Computing the weight enumerator of a linear code is exponential in the dimension of the code.
However the support splitting algorithm consists in computing the weight enumerator of hulls of codes. We are going to compute the hull of $N$ random codes and check that the dimension of the hull is usually very close to $0$.

**Q2.a.** Write a function ``Stat_Dimension_hull(q,n,k,N)`` that picks $N$ random codes of length $n$ and dimension $k$ and returns a dictionnary whose keys are the integers $0$ to $min(k,\lfloor n/2\rfloor)$ and the value associated to the key $m$ is the proportion of codes whose hull has dimension $m$.

*N.B.: You should use ``codes.random_linear_code(GF(q),n,k)`` to generate random codes.*

In [146]:
def Stat_Dimension_hull(q,n,k,N):
    d = {m:0 for m in range(min(k,floor(n/2)))}
    for _ in range(N) : 
        C = codes.random_linear_code(GF(q),n,k)
        hC = hull(C)
        m = hC.dimension()
        d[m] += 1/N     
    return d
    

print(Stat_Dimension_hull(4,11,7,1000))

{0: 18/25, 1: 67/250, 2: 3/250, 3: 0, 4: 0}


Sendrier computed the probability that a *random code* of dimension $k$ has a hull of dimension $m$. For $q$ large enough, we have

$$\mathbb{P}( \dim(\mathcal{H}(C))=m) \underset{n,k \rightarrow \infty}{\longrightarrow} R_m$$
where

$$R_0=\prod_{i \geq 1} \frac{1}{1+q^{-i}} \text{ and } R_i=\frac{R_{i-1}}{q^i-1}.$$

Here is a piece of code that computes $R_0$ for a value of $q$ with precision $10^{-16}$. Beware when printing this value, you should use ``float(R0(q))`` to get something "readable".

In [147]:
def R0(q):
    res=1/(1+1/q)
    i=2
    while true:
        new_res=res/(1+1/q^i)
        if abs(res-new_res)<10^(-16):
            return res
        else:
            res=new_res
            i=i+1
            
print(R0(2))
print (":(")
print(float(R0(2)))
print (":)")

1464786236049624260685376124451850855163464793406634804134924595650009137699196073488789740256022690528233381736519243125814513094767368490512041347117169397126296553786416376054862129417690612264236680596273173217497529585457669187441494533882414736478236103398964938092339616557313591147565568778390539250062537780210879899306233799589030501134232848228066226546715016482345028680908893189113380864/3492388794887583864920996229073791902212611916638104237188221670165896229938188698858885682944186125896472953995652089884626819671919218139008436799568381657503933698533107123294889960745276338053509724675640523257976741640489811484883668085796591946718771453572721586860968930199287172116629906727283028163944452229983774547150085037537936774212778086692307435334753446480213225339327181243896484375
:(
0.4194224417951078
:)


**Q2.b.** Create a function ``theoric_values(q,n,k)`` that returns the dictionnary of the form `i:R_i` for $i=0,\dots,min(k,\lfloor n/2 \rfloor)$ using ``R0(q)`` above.

In [148]:
def theoric_values(q,n,k):
    R=[R0(q)]
    for i in range(min(k, floor(n/2))):
        R.append(R[i]/(q^(i+1)-1))
    return R

print(float(R0(4)))
R=theoric_values(4,10,7)
for ri in R:
    print(float(ri))


0.7375122541538012
0.7375122541538012
0.24583741805126705
0.016389161203417803
0.00026014541592726673
1.0201781016755559e-06
9.972415461149128e-10


**Q2.c.** Check that the outputs of ``Stat_Dimension_hull`` is consistent with Sendrier's result. To do that, for every $i \in \{0,\dots,min(k,\lfloor n/2 \rfloor)\}$, check that the difference between the proportion of codes whose hulls has dimension $i$ you obtained and the theoretical value provided by Sendrier is bounded by $1/\sqrt{N}$.

You should test for $(q,n,k,N)=(25,60,20,100), (2,150,80,500), (4,100,40,500)$.

*N.B.: You should write $1/\sqrt{N}$= ``N^(-0.5)`` to avoid litteral computation. You should also use ``abs( )`` to compute some absolute value.*

In [149]:
def fonction_test(q,n,k, N):
    res=True
    stat=Stat_Dimension_hull(q,n,k,N)
    theoric=theoric_values(q,n,k)
    borne=min(k, floor(n/2))
    for i in range(borne):
        if abs(stat[i]-theoric[i])> N^(-0.5):
            res = False
    return res


q,n,k,N=25,60,20,100
print(fonction_test(q,n,k, N))

q,n,k,N=2,150,80,500
print(fonction_test(q,n,k, N))

q,n,k,N=4,100,40,500
print(fonction_test(q,n,k, N))



True
True
True


## 2. Support splitting algorithm
The support splitting algorithm requires computing punctured codes. To avoid dealing with shifted indices, we will keep the original length when puncturing, that is to say that for $I \subset \{0,\dots,n-1\}$, *the code $C$ punctured at $I$* is the image of $C$ by the map $\mathcal{Z}_I:\mathbb{F}_q^n \rightarrow \mathbb{F}_q^n$ defined by $\mathcal{Z}_I(x_1,\dots,x_n)=(x'_1,\dots,x'_n)$ with $x'_i=0$ if $i \in I$ and $x'_i=x_i$ if $i \notin I$.

**Q3.** Explain why the function `Puncture(C, I)` defined below exactly computes the code $C$ punctured at $I$.

In [150]:
def Puncture(C,I):
    G=C.generator_matrix()
    Cols=G.columns()
    n=C.length()
    k=C.dimension()
    z=vector(C.base_field(),k*[0]) # Une colonne nulle pour fixer les coordonnées dans I à zéro
    ColsGI=[]
    # ColsGI sera la matrice générant le code poinçonné
    for i in range(n):
        # On ajoute dans ColsGI la colonne nulle si i est dans I, donc la ième coordonnée est nulle
        if i in I: # Si la ième coordonnées est dans I on met la colonne nulle donc la ième coordonnée est fixée à zéro
            ColsGI.append(z)
        else: # Si la ième coordonées n'est pas dans I alors on la garde cad on met la colonne correspondante de G
            ColsGI.append(Cols[i])
    return codes.LinearCode(matrix(ColsGI).transpose())

In [151]:
#voir commentaires

**Q4.** Write a function `SupportSplitting`that takes as input a code $C$ and return a **dictionnary** formed as follows: a key is a pair (as a list) of weight enumerators (computed with `weight_enumerator_fixed` defined above) and the value associated is the list of indices $i$ such that these are the enumerators of the codes $\mathcal{H}(C_i)$ (punctured code at $I=\{i\}$) and $\mathcal{H}((C^\perp)_i)$.

In [152]:
def SupportSplitting(C):
    n=C.length()
    d={}
    Cdual=C.dual_code()
    for i in range(n):
        I=[i]
        Cponct=Puncture(C,I)
        hull1=hull(Cponct)
        
        Cdualponct=Puncture(Cdual,I)
        hull2=hull(Cdualponct)
        coord1=weight_enumerator_fixed(hull1)

        coord2=weight_enumerator_fixed(hull2)
        if (coord1, coord2) in d.keys():
            d[(coord1, coord2)].append(i)
        else :
            d[(coord1, coord2)] = [i]
    return d
    

We recall that a signature $S$ maps a code $C$ on length $n$ and an index $i \in \{0,\dots,n-1\}$ to some *value* (usually tuples of weight enumerators of some codes related to $C$ and $i$) and satisfies that for any code $C \subset \mathbb{F}_q$, any permutation $\sigma \in \mathcal{S}_n$ and any index $i \in \{0,\dots,n-1\}$, we have $S(\sigma(C),\sigma(i))=S(C,i)$.

The aim is to design a *fully discriminating signature* for $C$, that is to say a map $S:i \in \{0,\dots,n-1\} \mapsto S(i)$ such that $S(C,i)= S(C,j)$ if and only if $i=j$ by refining the Support Spliting algorithm.

We will work on the following code, defined over $\mathbb{F}_2$.

In [153]:
G2 = matrix(GF(2), [
[1,0,1,0,1,0,1,1,1,1],
[0,1,0,1,1,1,1,1,0,0],
[1,0,0,1,1,0,0,1,0,1],
[1,1,1,1,1,1,0,1,0,1],
[0,1,0,0,0,0,1,0,0,0],
[0,0,0,0,0,0,1,1,1,1],
[0,1,0,1,0,0,0,1,1,0],
[0,0,1,1,1,0,0,1,1,1],
[0,1,1,0,0,1,0,0,0,0],
[0,0,1,1,1,0,0,0,1,0]
])
G=block_matrix([[identity_matrix(GF(2),10),G2]])
C=codes.LinearCode(G)
#Correction
SSC=SupportSplitting(C)

print(SSC)

{(x^12 + 1, 1): [0, 1, 10, 19], (x^10 + 1, 1): [2, 3, 15, 17], (x^6 + 1, 1): [4, 16], (1, x^8 + 1): [5, 7, 14], (x^14 + 1, 1): [6], (x^8 + 1, 1): [8, 11, 12], (1, x^10 + 1): [9, 13, 18]}


**Q5.** Run the Support Splitting Algorithm (SSA) on this code. There should be a position that is fully discriminated, i.e., there exists $i_0 \in\{0,\dots,n-1\}$ such that 
$$(w(\mathcal{H}(C_{i_0})),w(\mathcal{H}((C^\perp)_{i_0})=(w(\mathcal{H}(C_{j})),w(\mathcal{H}((C^\perp)_{j}) \: \Rightarrow \: j=i_0.$$

In [154]:
SSC=SupportSplitting(C)


Now run the Support Splitting Algorithm on $C_{i_0}$, the code $C$ punctured at $i_0$. This should discriminate another position: there exists $i_1 \neq i_0$ such that the pair $(w(\mathcal{H}(C_{i_0,i_1})),w(\mathcal{H}((C_{i_0}^\perp)_{i_1})$ is "unique".

In [155]:
i0= [value for value in SSC.values() if len(list(value))==1][0][0]#to complete
print(i0)
C1=Puncture(C, [i0])
SSC1=SupportSplitting(C1)
print(SSC1)

6
{(1, 1): [0, 1, 3, 4, 5, 8, 11, 12, 13, 14, 15, 17, 18, 19], (x^14 + x^10 + x^8 + 1, x^14 + 1): [2], (x^14 + 1, x^14 + 1): [6], (x^14 + 1, x^14 + x^10 + x^8 + 1): [7, 9], (x^14 + x^12 + x^6 + 1, x^14 + 1): [10, 16]}


In [156]:
i1=[value for value in SSC1.values() if len(list(value))==1][0][0]
print(i1)
C2=Puncture(C, [i1])
SSC2=SupportSplitting(C2)
print(SSC2)

2
{(1, 1): [0, 1, 5, 7, 11, 13, 15, 16, 17, 18], (x^10 + 1, x^10 + 1): [2], (2*x^10 + x^8 + 1, x^10 + 1): [3, 8, 12], (x^12 + x^10 + x^6 + 1, x^10 + 1): [4, 19], (x^14 + x^10 + x^8 + 1, x^10 + 1): [6], (x^10 + 1, 2*x^10 + x^8 + 1): [9, 14], (x^12 + 2*x^10 + 1, x^10 + 1): [10]}


However, by combining the outputs of the SSA on $C$ and $C_{i_0}$, we can discriminate more position. Explain why.

On peut trouver faire un dictionnaire avec les quadruplet pour discrimer plus de gens (genre on peut a [7, 9] dans le deuxième et le 7 est sans le 9 sans)

In [ ]:
d_new = {}
for k1 in SSC.keys() :
    for k2 in SSC1.keys() :
        l = list(set(SSC[k1]) & set(SSC1[k2])) #intersection de deux listes
        if l != [] :
            d_new[k1+k2] = l
print(d_new.values())

dict_values([[0, 1, 19], [10], [17, 3, 15], [2], [4], [16], [5, 14], [7], [6], [8, 11, 12], [18, 13], [9]])


**Q6.** Find $i_2 \notin \{i_0,i_1\}$ such that the outputs of ``SupportSplitting`` on $C$, $C_{i_0}$, $C_{i_1}$ and $C_{i_2}$ enables you to fully discriminate every position. You can write a piece of code that helps you to process all these outputs.

Note : several values for $i_2$ are valid.

In [ ]:
i2=[value for value in d_new.values() if len(list(value))==1][0][0]
print(i2) 


#Ça c'est un i2 possible, il faut maintenant refaire la même chose avec tout les 4 Codes plus haut jusqu'à avoir un dico où toutes les valeurs sont discriminées)
#si le i2 suffit pas on prend un autre i2 possible

10


**Q7.** Here is a piece of code that generates a random permutation $P$ and create the associated code $D=CP$. Use the fully discriminating signature to recover this permutation. 

In [165]:
p = Permutations(20).random_element()
P=p.to_matrix()
G = C.generator_matrix()*P
D = codes.LinearCode(G)


In [160]:
#for your code